# Day 2: Sentiment Modeling and Model Selection

This notebook follows the Day 2 part of the internship PDF up to the dashboard stage. It trains NLP sentiment classifiers using employee survey comments, compares model performance, selects the best model, and exports the final TF-IDF pipeline.

## Important Dataset Note

The OSMI survey dataset does not include manually labeled sentiment classes. To complete the sentiment classification workflow, sentiment labels are generated from comment polarity using TextBlob:

- Positive: polarity > 0.05
- Negative: polarity < -0.05
- Neutral: otherwise

This is suitable for the internship NLP workflow, but it should not be treated as a clinical or production HR decision model.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "Notebooks" else Path.cwd().resolve()
sys.path.append(str(ROOT / "scripts"))

import joblib
import pandas as pd

from complete_until_dashboard import (
    DATA_DIR,
    MODELS_DIR,
    clean_text,
    sentiment_label,
    train_sentiment_models,
)

## Load Cleaned Survey Dataset

In [2]:
df = pd.read_csv(DATA_DIR / "cleaned_survey.csv")
df.shape

(1250, 27)

In [3]:
df[["Age", "Gender", "Country", "treatment", "work_interfere", "comments"]].head()

,Age,Gender,Country,treatment,work_interfere,comments
0,37,Male,United States,Yes,Often,NaN
1,44,Male,United States,No,Rarely,NaN
2,32,Male,Canada,No,Rarely,NaN
3,31,Male,United Kingdom,Yes,Often,NaN
4,31,Male,United States,No,Never,NaN


## Prepare Comment Data

In [4]:
comments_df = df[df["comments"].notna()].copy()
comments_df["clean_comments"] = comments_df["comments"].apply(clean_text)
comments_df = comments_df[comments_df["clean_comments"].str.len() > 0].copy()
comments_df["sentiment"] = comments_df["comments"].apply(sentiment_label)

comments_df[["comments", "clean_comments", "sentiment"]].head()

,comments,clean_comments,sentiment
13,I'm not on my company's health insurance which...,i m not on my company s health insurance which...,Positive
15,I have chronic low-level neurological issues t...,i have chronic low level neurological issues t...,Positive
16,My company does provide healthcare but not to ...,my company does provide healthcare but not to ...,Neutral
24,Relatively new job. Ask again later,relatively new job ask again later,Positive
25,Sometimes I think about using drugs for my me...,sometimes i think about using drugs for my men...,Positive


In [5]:
comments_df["sentiment"].value_counts()

sentiment
Positive    71
Negative    50
Neutral     39
Name: count, dtype: int64

## Train TF-IDF Sentiment Models

The training function compares:

- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM
- Tuned Logistic Regression

The final model is selected by highest Weighted F1 score, with Accuracy used as the secondary sorting metric.

In [6]:
model_info = train_sentiment_models(comments_df)
model_info["results"]

,Model,Accuracy,Weighted F1
0,Logistic Regression,0.425,0.419680
1,Tuned Logistic Regression,0.425,0.407239
2,Naive Bayes,0.500,0.374545
3,Linear SVM,0.375,0.361869


## Selected Model

In [7]:
print("Selected model:", model_info["best_name"])
print("\nClassification report:\n")
print(model_info["classification_report"])

Selected model: Logistic Regression

Classification report:

              precision    recall  f1-score   support

    Negative       0.55      0.50      0.52        12
     Neutral       0.11      0.10      0.11        10
    Positive       0.50      0.56      0.53        18

    accuracy                           0.42        40
   macro avg       0.39      0.39      0.38        40
weighted avg       0.42      0.42      0.42        40



## Export Final Sentiment Pipeline

The exported object is a complete scikit-learn Pipeline containing TF-IDF vectorization and the selected classifier.

In [8]:
model_path = MODELS_DIR / "sentiment_tfidf_pipeline.pkl"
joblib.dump(model_info["best_model"], model_path)
model_path

WindowsPath('D:/PROJECTS/Mental_Health_Sentiment_Analyzer/models/sentiment_tfidf_pipeline.pkl')

## Quick Prediction Check

In [9]:
loaded_model = joblib.load(model_path)
sample_comments = [
    "I feel anxious and burned out because of work pressure.",
    "My company is supportive and provides helpful mental health resources.",
]
loaded_model.predict(sample_comments)

array(['Positive', 'Positive'], dtype=object)

## Conclusion

Logistic Regression is selected because it has the highest Weighted F1 score among the tested sentiment models. This notebook completes the PDF's model training, evaluation, comparison, selection, and export steps before dashboard development.